# 🫁 Lung Cancer Detection — Final Notebook
### Models: ResNet50 | VGG16 | InceptionV3 | Basic CNN
### Dataset: IQ-OTH/NCCD Augmented Lung Cancer Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D,
    BatchNormalization, Conv2D, MaxPooling2D, Flatten
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50, VGG16, InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow version:", tf.__version__)
print("All imports successful!")

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Dataset Path (Augmented Dataset) ─────────────────────────────────────────
dataset_path = '/content/drive/MyDrive/Augmented IQ-OTHNCCD lung cancer dataset'

# Check distribution
print("Class Distribution:")
print("="*40)
total = 0
for class_name in sorted(os.listdir(dataset_path)):
    class_path = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        total += count
        print(f"  {class_name}: {count} images")
print(f"  Total: {total}")

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
IMG_SIZE_PRETRAINED = (224, 224)   # for ResNet50, VGG16, InceptionV3
IMG_SIZE_CNN = (128, 128)          # for Basic CNN
BATCH_SIZE = 32
NUM_CLASSES = 3
CLASS_NAMES = ['Benign cases', 'Malignant cases', 'Normal cases']

In [ ]:
# ── Data Loaders for Pretrained Models (224x224 RGB) ─────────────────────────
train_datagen_pretrained = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.2
)

test_datagen_pretrained = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen_pretrained = train_datagen_pretrained.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE_PRETRAINED,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='training',
    shuffle=True,
    seed=42
)

val_gen_pretrained = test_datagen_pretrained.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE_PRETRAINED,
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='validation',
    shuffle=False,
    seed=42
)

print("Pretrained model data loaders ready!")
print("Train samples:", train_gen_pretrained.samples)
print("Val samples:", val_gen_pretrained.samples)
print("Classes:", train_gen_pretrained.class_indices)

In [ ]:
# ── Data Loaders for Basic CNN (128x128 Grayscale) ───────────────────────────
train_datagen_cnn = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.15
)

test_datagen_cnn = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15
)

train_gen_cnn = train_datagen_cnn.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE_CNN,
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='training',
    shuffle=True
)

val_gen_cnn = train_datagen_cnn.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE_CNN,
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    subset='validation',
    shuffle=False
)

test_gen_cnn = test_datagen_cnn.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE_CNN,
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Basic CNN data loaders ready!")
print("Train samples:", train_gen_cnn.samples)
print("Val samples:", val_gen_cnn.samples)
print("Test samples:", test_gen_cnn.samples)

In [ ]:
# ── Class Weights ─────────────────────────────────────────────────────────────
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen_pretrained.classes),
    y=train_gen_pretrained.classes
)
CLASS_WEIGHTS = dict(enumerate(weights))
print("Class weights:", CLASS_WEIGHTS)

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────
def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor='val_accuracy',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=4,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            f'{model_name}_best.keras',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

In [ ]:
# ── Plot and Evaluate Helpers ─────────────────────────────────────────────────
def plot_history(history, model_name):
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history.get('accuracy', []), label='Train Accuracy', linewidth=2)
    plt.plot(history.history.get('val_accuracy', []), label='Val Accuracy', linewidth=2)
    plt.title(f'{model_name} — Accuracy', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(history.history.get('loss', []), label='Train Loss', linewidth=2)
    plt.plot(history.history.get('val_loss', []), label='Val Loss', linewidth=2)
    plt.title(f'{model_name} — Loss', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{model_name}_history.png', dpi=150)
    plt.show()


def evaluate_pretrained(model, model_name, val_gen):
    """Evaluate pretrained models using ImageDataGenerator"""
    val_gen.reset()
    preds = model.predict(val_gen, verbose=1)
    y_pred = np.argmax(preds, axis=1)
    y_true = val_gen.classes

    acc = np.mean(y_true == y_pred) * 100
    print(f"\n{'='*50}")
    print(f"  {model_name} — Test Accuracy: {acc:.2f}%")
    print(f"{'='*50}\n")

    labels = list(val_gen.class_indices.keys())
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'{model_name} Confusion Matrix (Acc: {acc:.1f}%)', fontsize=13)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'{model_name}_confusion.png', dpi=150)
    plt.show()

    print(classification_report(y_true, y_pred, target_names=labels))
    return acc


def evaluate_cnn(model, model_name, test_gen):
    """Evaluate Basic CNN model"""
    label_map = dict((v, k) for k, v in test_gen.class_indices.items())
    labels = [label_map[i] for i in range(NUM_CLASSES)]

    test_gen.reset()
    score = model.evaluate(test_gen, verbose=1)
    print(f'Test accuracy: {score[1]*100:.2f}%')

    test_gen.reset()
    preds = model.predict(test_gen)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes

    acc = score[1] * 100
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f'{model_name} Confusion Matrix (Acc: {acc:.1f}%)', fontsize=13)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'{model_name}_confusion.png', dpi=150)
    plt.show()

    print(classification_report(y_true, y_pred, target_names=labels))
    return acc

print("Helper functions ready!")

## 🔵 ResNet50 — Two-Phase Training

In [ ]:
# ── ResNet50 ──────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  RESNET50")
print("="*60)

base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_resnet.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_resnet.output)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

resnet_model = Model(inputs=base_resnet.input, outputs=output)
resnet_model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Phase 1: Training custom head (5 epochs) ---")
resnet_model.fit(train_gen_pretrained, validation_data=val_gen_pretrained,
                 epochs=5, class_weight=CLASS_WEIGHTS, verbose=1)

# Phase 2 - Unfreeze last 30 layers
for layer in base_resnet.layers[:-30]:
    layer.trainable = False
for layer in base_resnet.layers[-30:]:
    layer.trainable = True

resnet_model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Phase 2: Fine-tuning last 30 layers (50 epochs max) ---")
history_resnet = resnet_model.fit(
    train_gen_pretrained,
    validation_data=val_gen_pretrained,
    epochs=50,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks('ResNet50'),
    verbose=1
)

resnet_model.save('ResNet50.keras')
shutil.copy('ResNet50_best.keras', '/content/drive/MyDrive/ResNet50_best.keras')
print(" ResNet50 saved to Drive!")

plot_history(history_resnet, 'ResNet50')
resnet_acc = evaluate_pretrained(resnet_model, 'ResNet50', val_gen_pretrained)

## 🟢 VGG16 — Two-Phase Training

In [ ]:
# ── VGG16 ─────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  VGG16")
print("="*60)

base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_vgg.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_vgg.output)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

vgg_model = Model(inputs=base_vgg.input, outputs=output)
vgg_model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Phase 1: Training custom head (5 epochs) ---")
vgg_model.fit(train_gen_pretrained, validation_data=val_gen_pretrained,
              epochs=5, class_weight=CLASS_WEIGHTS, verbose=1)

# Phase 2 - Unfreeze block4 and block5
for layer in base_vgg.layers:
    if 'block5' in layer.name or 'block4' in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

vgg_model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Phase 2: Fine-tuning block4 + block5 (50 epochs max) ---")
history_vgg = vgg_model.fit(
    train_gen_pretrained,
    validation_data=val_gen_pretrained,
    epochs=50,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks('VGG16'),
    verbose=1
)

vgg_model.save('VGG16.keras')
shutil.copy('VGG16_best.keras', '/content/drive/MyDrive/VGG16_best.keras')
print(" VGG16 saved to Drive!")

plot_history(history_vgg, 'VGG16')
vgg_acc = evaluate_pretrained(vgg_model, 'VGG16', val_gen_pretrained)

## 🟣 InceptionV3 — Two-Phase Training

In [ ]:
# ── InceptionV3 ───────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  INCEPTIONV3")
print("="*60)

base_inception = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_inception.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_inception.output)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

inception_model = Model(inputs=base_inception.input, outputs=output)
inception_model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Phase 1: Training custom head (5 epochs) ---")
inception_model.fit(train_gen_pretrained, validation_data=val_gen_pretrained,
                    epochs=5, class_weight=CLASS_WEIGHTS, verbose=1)

# Phase 2 - Unfreeze last 50 layers
for layer in base_inception.layers[:-50]:
    layer.trainable = False
for layer in base_inception.layers[-50:]:
    layer.trainable = True

inception_model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n--- Phase 2: Fine-tuning last 50 layers (50 epochs max) ---")
history_inception = inception_model.fit(
    train_gen_pretrained,
    validation_data=val_gen_pretrained,
    epochs=50,
    class_weight=CLASS_WEIGHTS,
    callbacks=get_callbacks('InceptionV3'),
    verbose=1
)

inception_model.save('InceptionV3.keras')
shutil.copy('InceptionV3_best.keras', '/content/drive/MyDrive/InceptionV3_best.keras')
print(" InceptionV3 saved to Drive!")

plot_history(history_inception, 'InceptionV3')
inception_acc = evaluate_pretrained(inception_model, 'InceptionV3', val_gen_pretrained)

## 🟡 Advance CNN — High Accuracy Custom Model
**Uses:** 128×128 Grayscale images



In [ ]:
# ── Basic CNN ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  ADVANCED CNN (Hybrid Model)")
print("="*60)

cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu',
           input_shape=(128, 128, 1)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

# Class weights for CNN
cnn_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen_cnn.classes),
    y=train_gen_cnn.classes
)
cnn_class_weights = dict(enumerate(cnn_weights))

# Callbacks for CNN
cnn_callbacks = [
    EarlyStopping(monitor='val_loss', mode='min', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', patience=3,
                     factor=0.3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('BasicCNN_best.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print("\n--- Training Basic CNN (50 epochs max) ---")
history_cnn = cnn_model.fit(
    train_gen_cnn,
    epochs=50,
    validation_data=val_gen_cnn,
    class_weight=cnn_class_weights,
    callbacks=cnn_callbacks,
    verbose=1
)

cnn_model.save('advanced_cnn.keras')
shutil.copy('BasicCNN_best.keras', '/content/drive/MyDrive/advanced_cnn_best.keras')
print(" Advanced CNN saved to Drive!")

plot_history(history_cnn, 'AdvancedCNN')
cnn_acc = evaluate_cnn(cnn_model, 'AdvancedCNN', test_gen_cnn)

In [ ]:
# ── Sample Predictions Visualization (like lung-cancer-cnn notebook) ──────────
label_map = dict((v, k) for k, v in test_gen_cnn.class_indices.items())

# Rebuild test data
x_test = []
y_test = []
for i in range(len(test_gen_cnn)):
    x_batch, y_batch = test_gen_cnn[i]
    x_test.append(x_batch)
    y_test.append(y_batch)
x_test = np.concatenate(x_test)
y_test = np.concatenate(y_test)
y_true_vis = np.argmax(y_test, axis=1) if y_test.ndim == 2 else y_test

test_gen_cnn.reset()
preds_vis = cnn_model.predict(test_gen_cnn)
y_pred_vis = np.argmax(preds_vis, axis=1)

# Show sample predictions
plt.figure(figsize=(12, 10))
num_examples = 3
classes_ids = sorted(label_map.keys())
i = 1

for c in classes_ids:
    idxs = np.where(y_true_vis == c)[0][:num_examples]
    for idx in idxs:
        plt.subplot(len(classes_ids), num_examples, i)
        img = x_test[idx].squeeze()
        pred_class = label_map[y_pred_vis[idx]]
        prob = np.max(preds_vis[idx])
        true_class = label_map[y_true_vis[idx]]
        plt.imshow(img, cmap='gray')
        plt.title(f'Pred: {pred_class} ({prob:.0%})\nTrue: {true_class}',
                  fontsize=8,
                  color='green' if pred_class == true_class else 'red')
        plt.axis('off')
        i += 1

plt.suptitle('Basic CNN — Sample Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('BasicCNN_samples.png', dpi=150)
plt.show()

In [ ]:
# ── Final Accuracy Summary ────────────────────────────────────────────────────
print("\n" + "="*60)
print("  FINAL ACCURACY SUMMARY")
print("="*60)
print(f"  ResNet50   : {resnet_acc:.2f}%")
print(f"  VGG16      : {vgg_acc:.2f}%")
print(f"  InceptionV3: {inception_acc:.2f}%")
print(f"  Basic CNN  : {cnn_acc:.2f}%")
print("="*60)

# Bar chart comparison
models = ['ResNet50', 'VGG16', 'InceptionV3', 'Basic CNN']
accs = [resnet_acc, vgg_acc, inception_acc, cnn_acc]
colors = ['#4A90D9', '#5CB85C', '#9B59B6', '#E67E22']

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accs, color=colors, edgecolor='black', linewidth=0.8)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', va='bottom',
             fontweight='bold', fontsize=12)
plt.ylim(0, 110)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Model Accuracy Comparison — Lung Cancer Detection', fontsize=14)
plt.axhline(y=95, color='red', linestyle='--', alpha=0.5, label='95% target')
plt.legend()
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150)
plt.show()

# Save all to Drive
print("\nSaving all models to Google Drive...")
for f in ['ResNet50_best.keras', 'VGG16_best.keras',
          'InceptionV3_best.keras', 'BasicCNN_best.keras']:
    try:
        shutil.copy(f, f'/content/drive/MyDrive/{f}')
        print(f" {f} saved!")
    except:
        print(f" {f} not found!")